# LINEAR vs MLP REGRESSION EXPERIMENT (PyTorch)

## Overview

In this notebook, we compare different neural network models on a simple regression problem using PyTorch.

The goal is to understand:

- the difference between linear and nonlinear models,
- how model capacity affects learning,
- why overfitting happens,
- and how regularization improves generalization.

We train and evaluate four models:

1. **Linear Regression** — a simple linear baseline.
2. **Small MLP** — a small neural network with nonlinear activations.
3. **Large MLP** — a much higher-capacity network that can overfit.
4. **Regularized MLP** — the same large network trained with regularization.

The notebook follows a complete machine learning workflow:
- data loading,
- preprocessing,
- training,
- validation,
- model selection,
- and final testing.

---

## Pipeline Summary

The experiment follows these steps:

1. Load the training and test datasets.
2. Split the training data into train and validation subsets.
3. Normalize the data using statistics computed only from the training split.
4. Create PyTorch datasets and dataloaders.
5. Train multiple models using gradient descent.
6. Monitor training and validation performance.
7. Compare models using RMSE and MAE.
8. Select the best model using validation performance.
9. Evaluate the selected model on the test set.

---

## Key Ideas

This notebook demonstrates several important deep learning concepts:

- **Model capacity**: larger models can represent more complex functions.
- **Nonlinearity**: MLPs can learn nonlinear relationships using activation functions.
- **Generalization**: strong validation performance matters more than memorizing training data.
- **Regularization**: techniques like weight decay and early stopping help reduce overfitting.
- **Reproducibility**: fixing random seeds ensures consistent results.

---

## Output

The notebook generates:

- Training and validation loss curves
- Model comparison plots
- RMSE and MAE metrics
- Predicted regression curves
- Final test set evaluation


## Dependencies and Environment Setup

We import:

- `torch` for neural networks and optimization,
- `numpy` and `pandas` for numerical data processing,
- `matplotlib` for visualization,
- and `sklearn` for dataset splitting utilities.


In [ ]:
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split


## Reproducibility Setup

Machine learning experiments contain randomness:
- parameter initialization,
- data shuffling,
- and GPU operations.

To make results reproducible, we fix the random seeds for NumPy and PyTorch.

This allows the notebook to produce the same results across runs.


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


In [ ]:
seed_everything(42)


## Hardware Selection

PyTorch can run computations on:
- the CPU,
- or a CUDA-enabled GPU.

If a GPU is available, tensors and models are moved to the GPU for faster computation.

Otherwise, the notebook falls back to the CPU.


In [ ]:
seed_everything()

device = torch.device(
         "cuda" if torch.cuda.is_available()
    else "mps"  if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")


## Dataset Paths

We use `pathlib.Path` to define platform-independent file paths for the dataset directory.


In [ ]:
DATA_DIR = Path("data/regression")


## Normalization Utilities

Neural networks usually train more effectively when inputs are centered and scaled.

We use **Z-score normalization**:


$x_{norm} = \frac{x-\mu}{\sigma}$


where:
- $\mu$ is the mean,
- $\sigma$ is the standard deviation.

We also define a `denormalize` function to convert predictions back to the original scale.


In [ ]:
def normalize(x, mean, std):
    return (x - mean) / std


def denormalize(x, mean, std):
    return (x * std) + mean


## Custom PyTorch Dataset

We create a custom `Dataset` class to:

- store input and target values,
- apply normalization,
- convert data into tensors,
- and make the dataset compatible with PyTorch `DataLoader`s.

This keeps preprocessing logic organized and reusable.


In [ ]:
class RegressionDataset(Dataset):
    def __init__(
        self,
        df,
        normalize_x=False,
        normalize_y=False,
        x_mean=None,
        x_std=None,
        y_mean=None,
        y_std=None
    ):

        self.raw_x = df["x"].values.astype(np.float32).reshape(-1, 1)
        self.raw_y = df["y"].values.astype(np.float32).reshape(-1, 1)
        x = self.raw_x.copy()
        y = self.raw_y.copy()

        if normalize_x:
            if x_mean is None or x_std is None:
                raise ValueError("x_mean and x_std must be provided when normalize_x=True")
            x = normalize(x, x_mean, x_std)

        if normalize_y:
            if y_mean is None or y_std is None:
                raise ValueError("y_mean and y_std must be provided when normalize_y=True")
            y = normalize(y, y_mean, y_std)

        self.x_mean = x_mean
        self.x_std = x_std
        
        self.y_mean = y_mean
        self.y_std = y_std

        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


## Train / Validation / Test Split

We load the dataset from CSV files and split the original training data into:

- a **training split** used for optimization,
- and a **validation split** used to monitor generalization.

The test set remains untouched until the very end of the experiment.


In [ ]:
train_full_df = pd.read_csv(DATA_DIR / "train" / "train_dataset.csv")

test_df = pd.read_csv(DATA_DIR / "test" / "test_dataset.csv")

train_df, val_df = train_test_split(
    train_full_df,
    test_size=0.25,
    random_state=42
)

print(f"\nTrain split size:      {len(train_df)}")
print(f"Validation split size: {len(val_df)}")
print(f"Test size:             {len(test_df)}")


## Compute Training Statistics

Normalization statistics must be computed using only the training split.

This avoids **data leakage**, where information from validation or test data accidentally influences training.

We compute:
- the mean,
- and the standard deviation

for both inputs and targets.

A small epsilon value is added to prevent division-by-zero errors.

Normalizing targets can also improve optimization stability because the network predicts values on a more consistent numerical scale.


In [ ]:
EPS = 1e-8

x_mean = train_df["x"].mean()
x_std = train_df["x"].std(ddof=0) + EPS

y_mean = train_df["y"].mean()
y_std = train_df["y"].std(ddof=0) + EPS

print(f"\nTrain x mean: {x_mean:.4f}")
print(f"Train x std:  {x_std:.4f}")
print(f"\nTrain y mean: {y_mean:.4f}")
print(f"Train y std:  {y_std:.4f}")


## Create Dataset Objects

We now create:
- training,
- validation,
- and test datasets.

The normalization statistics computed from the training split are reused consistently across all datasets.

This ensures that validation and test data are transformed in the same way as training data.


In [ ]:
train_dataset = RegressionDataset(
    train_df,
    normalize_x=True,
    normalize_y=True,
    x_mean=x_mean,
    x_std=x_std,
    y_mean=y_mean,
    y_std=y_std
)

val_dataset = RegressionDataset(
    val_df,
    normalize_x=True,
    normalize_y=True,
    x_mean=x_mean,
    x_std=x_std,
    y_mean=y_mean,
    y_std=y_std
)

test_dataset = RegressionDataset(
    test_df,
    normalize_x=True,
    normalize_y=True,
    x_mean=x_mean,
    x_std=x_std,
    y_mean=y_mean,
    y_std=y_std
)


## Create DataLoaders

`DataLoader`s generate mini-batches during training.

Benefits of mini-batching:
- lower memory usage,
- faster optimization,
- and more stable gradient estimates.

We shuffle the training loader so the model does not see samples in the same order every epoch.


In [ ]:
BATCH_SIZE = 64
pin_memory = (device.type == "cuda")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True, pin_memory=pin_memory, 
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False, pin_memory=pin_memory, 
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False, pin_memory=pin_memory, 
)


## Training and Evaluation Functions

We define reusable functions for:
- one training epoch,
- model evaluation,
- and full training loops.

The training loop:
- computes losses,
- updates parameters,
- tracks validation performance,
- and optionally applies early stopping.

The best model weights are saved during training.


In [ ]:
def train_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0
    dataset_size = 0
    for xb, yb in loader:
        dataset_size += xb.size(0)
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        
    return total_loss / dataset_size


In [ ]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0
    dataset_size = 0
    with torch.no_grad():
        for xb, yb in loader:
            dataset_size += xb.size(0)
            xb = xb.to(device)
            yb = yb.to(device)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            total_loss += loss.item() * xb.size(0)
            
    return total_loss / dataset_size


In [ ]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn,
    max_epochs: int,
    patience: int | None
):

    model = model.to(device)
    train_losses = []
    val_losses = []
    best_val_loss = float("inf")
    best_state = None
    # Enable early stopping only if patience is provided
    use_early_stopping = patience is not None
    patience_counter = 0

    for epoch in range(max_epochs):
        
        train_loss = train_epoch(model, train_loader, loss_fn, optimizer)
        val_loss = evaluate(model, val_loader, loss_fn)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        # Save best model
        if use_early_stopping:
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

        # Print progress
        if epoch % 25 == 0:
            print(f"Epoch {epoch:4d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

        # Early stopping
        if use_early_stopping and patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch}")
            break

        # Save final state if no early stopping is used
        if not use_early_stopping:
            # Without early stopping, keep the final epoch weights
            best_state = copy.deepcopy(model.state_dict())
            best_val_loss = val_loss

    model.load_state_dict(best_state)
    return train_losses, val_losses, best_val_loss


## Model Architectures

We define two model families:

### Linear Regression

A single linear layer:


$y = Wx + b$


This model can only learn linear relationships.

### Multi-Layer Perceptron (MLP)

The MLP introduces hidden layers and nonlinear activation functions:


$f(x)=W_3\ \mathrm{ReLU}(W_2\ \mathrm{ReLU}(W_1x+b_1)+b_2)+b_3$


The ReLU activation:


$\text{ReLU}(x)=\max(0,x)$


allows the network to model nonlinear functions.

We create:
- a small MLP,
- a large MLP,
- and a regularized version of the large MLP.

In [ ]:
class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)


In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.hidden_size = hidden_size
        self.fc1 = nn.Linear(1, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x


In [ ]:
# Create models
lin_mod = LinearRegression()
sml_mlp = MLP(hidden_size=16)
lrg_mlp = MLP(hidden_size=512) 
reg_mlp = MLP(hidden_size=512)
# Intentionally oversized model to demonstrate overfitting


## Counting Parameters

The number of trainable parameters controls model capacity.

Larger models can represent more complex functions, but they are also more likely to overfit.

For a linear layer:

$\text{Parameters}=(\text{input_features}\times \text{output_features})+\text{output_features (bias)}$


| Layer     | Calculation           | Parameters |
|-----------|-----------------------|------------|
| **fc1**   | (1 x 16) + 16  | 32         |
| **fc2**   | (16 x 16) + 16 | 272        |
| **fc3**   | (16 x 1) + 1   | 17         |

We compare the parameter counts across all models to better understand their complexity.

> As parameter count increases, the model gains flexibility but also becomes more capable of memorizing noise.


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


In [ ]:
print(f"\nLinear Regression params: {count_parameters(lin_mod)}")
print(f"\nSmall MLP params: {count_parameters(sml_mlp)}")
print(f"\nLarge MLP params: {count_parameters(lrg_mlp)}")
print(f"\nRegularized MLP params: {count_parameters(reg_mlp)}")


In [ ]:
print(sml_mlp)

## Training the Models

We train all models using:
- Mean Squared Error (MSE) loss,
- and gradient-based optimization.

The regularized model uses:
- **AdamW** for weight decay regularization,
- and **early stopping** to reduce overfitting.

Regularization does not reduce the expressive power of the network.

Instead, it constrains how the model uses that capacity.


>**Note**: AdamW applies weight decay directly to parameters instead of through the loss, decoupling regularization from Adam’s adaptive gradient scaling and resulting in more consistent regularization behavior.


In [ ]:
criterion = nn.MSELoss()

lr = 1e-3

opt_lin_mod = torch.optim.Adam(lin_mod.parameters(),  lr=lr)
opt_sml_mlp = torch.optim.Adam(sml_mlp.parameters(),  lr=lr)
opt_lrg_mlp = torch.optim.Adam(lrg_mlp.parameters(),  lr=lr)
opt_reg_mlp = torch.optim.AdamW(reg_mlp.parameters(), lr=lr, weight_decay=1e-3)


This synthetic dataset is very small, so unusually large epoch counts are feasible here.

**Here `max_epochs=20000` is just to show overfitting**

> A smaller epoch count would normally be sufficient, but longer training is used here to intentionally visualize overfitting dynamics.

In [ ]:

print("\nTRAINING LINEAR MODEL")
lin_train_losses, lin_val_losses, lin_best_val_loss = train_model(lin_mod, 
                                                                  train_loader, val_loader, 
                                                                  opt_lin_mod, criterion,
                                                                  max_epochs=20000,patience=None)

print("\nTRAINING SMALL MLP")
sml_train_losses, sml_val_losses, sml_best_val_loss = train_model(sml_mlp, 
                                                                  train_loader, val_loader, 
                                                                  opt_sml_mlp, criterion,
                                                                  max_epochs=20000,patience=None)

print("\nTRAINING LARGE MLP")
lrg_train_losses, lrg_val_losses, lrg_best_val_loss = train_model(lrg_mlp, 
                                                                  train_loader, val_loader, 
                                                                  opt_lrg_mlp, criterion,
                                                                  max_epochs=20000,patience=None)

print("\nTRAINING REGULARIZED MLP")
reg_train_losses, reg_val_losses, reg_best_val_loss = train_model(reg_mlp,
                                                                  train_loader, val_loader,
                                                                  opt_reg_mlp, criterion,
                                                                  max_epochs=20000,patience=500)


## Learning Curves

We visualize:
- training loss,
- and validation loss

across epochs.

These curves help diagnose:
- underfitting,
- overfitting,
- and optimization behavior.

A logarithmic scale makes long-term training behavior easier to observe.

In [ ]:
plt.figure(figsize=(12, 6))
# Linear model
plt.plot(lin_train_losses, label="Linear Train Loss")
plt.plot(lin_val_losses, label="Linear Validation Loss")
# Small MLP
plt.plot(sml_train_losses, label="Small MLP Train Loss")
plt.plot(sml_val_losses, label="Small MLP Validation Loss")
# Large MLP
plt.plot(lrg_train_losses, label="Large MLP Train Loss")
plt.plot(lrg_val_losses, label="Large MLP Validation Loss")
# Regularized MLP
plt.plot(reg_train_losses, label="Regularized MLP Train Loss")
plt.plot(reg_val_losses, label="Regularized MLP Validation Loss")
plt.yscale("log")
plt.xscale("log")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss (Log Scale)")
plt.title("Learning Curves")
plt.legend()
plt.show()


## Prediction Utility

We create a reusable prediction function that:

1. normalizes inputs,
2. runs inference,
3. and converts predictions back to the original scale.

This keeps inference logic clean and consistent.


In [ ]:
def predict(model, x_numpy, x_mean, x_std, y_mean, y_std):
    dev = next(model.parameters()).device
    model.eval()
    x_numpy = normalize(x_numpy, x_mean, x_std)
    x_tensor = torch.tensor(x_numpy, dtype=torch.float32, device=dev)

    with torch.no_grad():
        preds = model(x_tensor)

    preds = preds.cpu().numpy()
    preds = denormalize(preds, y_mean, y_std)

    return preds


## Generate Regression Curves

We sample evenly spaced inputs across the domain and generate predictions from each model.

This allows us to visualize the function learned by each network.

In [ ]:
x_plot = np.linspace(-1, 1, 400).reshape(-1, 1).astype(np.float32)

lin_preds = predict(lin_mod, x_plot, x_mean, x_std, y_mean, y_std)
sml_preds = predict(sml_mlp, x_plot, x_mean, x_std, y_mean, y_std)
lrg_preds = predict(lrg_mlp, x_plot, x_mean, x_std, y_mean, y_std)
reg_preds = predict(reg_mlp, x_plot, x_mean, x_std, y_mean, y_std)


## Evaluation in Original Units

Training occurs in normalized space, but metrics computed in normalized space are difficult to interpret physically, so predictions and targets are converted back to the original scale before evaluation.

We therefore:
- denormalize predictions,
- denormalize targets,
- and compute RMSE and MAE using the original units.

RMSE penalizes large errors more strongly than MAE.


In [ ]:
def evaluate_real_world(model, loader, y_mean, y_std):
    dev = next(model.parameters()).device
    model.eval()

    total_mse = 0
    total_mae = 0
    dataset_size = 0

    with torch.no_grad():
        for xb, yb in loader:
            
            dataset_size += xb.size(0)
            
            xb = xb.to(dev)
            yb = yb.to(dev)
            preds = model(xb)
            
            preds_real = denormalize(preds, y_mean, y_std)
            yb_real    = denormalize(yb,    y_mean, y_std)

            batch_mse = nn.functional.mse_loss(preds_real, yb_real, reduction="sum")
            batch_mae = nn.functional.l1_loss(preds_real,  yb_real, reduction="sum")

            total_mse += batch_mse.item()
            total_mae += batch_mae.item()

    final_mse = total_mse / dataset_size
    final_rmse = np.sqrt(final_mse)
    final_mae = total_mae / dataset_size
    
    return final_rmse, final_mae


## Validation Performance

We evaluate all models on:
- the training split,
- and the validation split.

This comparison helps us understand:
- model fit,
- overfitting,
- and generalization quality.

**The test set is still untouched at this stage.**


In [ ]:
models = {
    "Linear Model": lin_mod,
    "Small MLP": sml_mlp,
    "Large MLP": lrg_mlp,
    "Regularized MLP": reg_mlp
}

metrics = {}


In [ ]:
print("PERFORMANCE ON TRAIN SET")

for model_name, model in models.items():

    train_rmse, train_mae = evaluate_real_world(
        model,
        train_loader,
        y_mean,
        y_std
    )

    metrics[model_name] = {"train_rmse": train_rmse, "train_mae": train_mae}

    print(f"\n{model_name}")
    print(f"Train RMSE: {train_rmse:.4f}")
    print(f"Train MAE : {train_mae:.4f}")


In [ ]:
print("PERFORMANCE ON VALIDATION SET")

for model_name, model in models.items():

    val_rmse, val_mae = evaluate_real_world(
        model,
        val_loader,
        y_mean,
        y_std
    )

    metrics[model_name]["val_rmse"] = val_rmse
    metrics[model_name]["val_mae"] = val_mae

    print(f"\n{model_name}")
    print(f"Validation RMSE: {val_rmse:.4f}")
    print(f"Validation MAE : {val_mae:.4f}")


## Comparing Model Predictions

We plot:
- the dataset samples,
- and the prediction curve of each model.

This makes it easier to visually compare:
- linear behavior,
- nonlinear fitting,
- and overfitting.

> **Test set is untouched until final evaluation**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 6), sharey=True)
axes = axes.ravel()
titles = [
    f"1. Linear Model\n"
    f"Train RMSE: {metrics['Linear Model']['train_rmse']:.2f} | "
    f"Val RMSE: {metrics['Linear Model']['val_rmse']:.2f}",

    f"2. Small MLP ({sml_mlp.hidden_size})\n"
    f"Train RMSE: {metrics['Small MLP']['train_rmse']:.2f} | "
    f"Val RMSE: {metrics['Small MLP']['val_rmse']:.2f}",
    
    f"3. Large MLP ({lrg_mlp.hidden_size})\n"
    f"Train RMSE: {metrics['Large MLP']['train_rmse']:.2f} | "
    f"Val RMSE: {metrics['Large MLP']['val_rmse']:.2f}",

    f"4. Regularized MLP ({reg_mlp.hidden_size})\n"
    f"Train RMSE: {metrics['Regularized MLP']['train_rmse']:.2f} | "
    f"Val RMSE: {metrics['Regularized MLP']['val_rmse']:.2f}"
]

predictions = [lin_preds, sml_preds, lrg_preds, reg_preds ]

colors = ['#e74c3c', '#9b59b6', '#2ecc71', '#f39c12']

for i, ax in enumerate(axes):
    
    # Train data
    ax.scatter(train_df["x"], train_df["y"], color='#34495e', alpha=0.3, s=30, label='Train Data')
    
    # Validation data
    ax.scatter(val_df["x"], val_df["y"], color='#3498db', marker='D', alpha=0.6, s=35, label='Val Data')

    # Model predictions
    ax.plot(x_plot, predictions[i], color=colors[i], linewidth=3, label='Model Prediction')

    # Styling
    ax.set_title(titles[i], fontsize=10, fontweight='bold', pad=10)
    ax.set_xlabel('Input (x)')
    ax.grid(True, linestyle=':', alpha=0.6)

    if i == 0:
        ax.set_ylabel('Target (y)')

    ax.legend(loc='best', fontsize=9)

plt.tight_layout()
plt.show()


## Selecting the Best Model

We select the final model using validation RMSE.

The test set is intentionally excluded from this decision to preserve an unbiased final evaluation.

The regularized model often performs better because regularization encourages smoother and more stable solutions.

> **Test set is untouched until final evaluation**

In [ ]:
best_model_name = min(
    metrics,
    key=lambda model_name: metrics[model_name]["val_rmse"]
)

best_model = models[best_model_name]

print("MODEL SELECTION")
print(f"Selected model: {best_model_name}")
print(f"Validation RMSE: {metrics[best_model_name]['val_rmse']:.4f}")


## Final Test Evaluation

After model selection, we evaluate the chosen model on the held-out test set.

This provides the most reliable estimate of real-world performance.


In [ ]:
test_rmse, test_mae = evaluate_real_world(best_model, test_loader, y_mean, y_std)

print("FINAL TEST PERFORMANCE")
print(f"Selected model: {best_model_name}")
print(f"Test RMSE: {test_rmse:.4f} (Penalizes large errors more strongly)")
print(f"Test MAE:  {test_mae:.4f} (Average absolute prediction error)")


## Generate Final Predictions

We generate predictions from the selected model across the input domain.

These predictions will be visualized alongside the train, validation, and test data.


In [ ]:
test_predictions = predict(best_model, x_plot, x_mean, x_std, y_mean, y_std)


## Final Visualization

We compare:
- the training data,
- validation data,
- test data,
- and the final prediction curve.

This plot provides a final visual check of generalization performance.


In [ ]:
plt.figure(figsize=(6, 4))

# Train data
plt.scatter(train_df["x"], train_df["y"], color='#34495e', alpha=0.3, s=30, label='Train Data')

# Validation data
plt.scatter(val_df["x"], val_df["y"], color='#3498db', marker='D', alpha=0.6, s=35, label='Val Data')

# Test data
plt.scatter(test_df["x"], test_df["y"], color='#2ecc71', marker='s', alpha=0.7, s=40, label='Test Data')


# Best model prediction
plt.plot(x_plot, test_predictions, color='#f39c12', linewidth=3, label=f'{best_model_name} Prediction')

# Styling
plt.title(f"{best_model_name}\nTest RMSE: {test_rmse:.2f} | Test MAE: {test_mae:.2f}",
          fontsize=10, fontweight='bold', pad=10)

plt.xlabel('Input (x)')
plt.ylabel('Target (y)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()


## Conclusion

This experiment demonstrates several core ideas in deep learning:

- Linear models are limited to linear relationships.
- MLPs can learn nonlinear functions through hidden layers and activations.
- Larger models have greater capacity but are more prone to overfitting.
- Regularization improves generalization by constraining parameter growth.
- Validation performance is essential for model selection.

More broadly, the notebook illustrates one of the central ideas of machine learning:

> Increasing model capacity improves flexibility, but good generalization requires regularization and careful evaluation.
